# Polaris RE — Validation & Benchmark Report

This notebook is the **diligence-grade** demonstration that Polaris RE
reproduces authoritative actuarial reference values. It runs the same
engine-agnostic validation pack that backs `polaris benchmark`
(`polaris_re.analytics.validation`) and renders the pass/fail table a
reviewer can read: *here is the reference, here is the engine, here is the
(machine-precision) difference.*

The references are **identities or cited constants, never recalled numbers**:

* **Closed-form** — constant-force term-insurance and temporary annuity-due
  APVs vs their exact discrete geometric-series closed forms.
* **Textbook** — the same term APV vs the continuous constant-force identity
  (Bowers et al., *Actuarial Mathematics* 2e, §4.2), within a documented
  monthly-discretisation tolerance.
* **Statutory deck** — the SOA Illustrative Life Table whole-life $A_x$,
  $\ddot a_x$, and net level premium $P_x$ at $i=6\%$ (App. 2A), reproduced
  by the live WholeLife engine to machine precision.
* **Experience improvement** — a *recovery identity*: a known mortality-
  improvement surface injected into a synthetic, ILEC-shaped experience
  extract (fed through the real `load_ilec` loader) is recovered by the
  tensor MI GAM to numerical precision.

Every code cell embeds its checks as `assert`s, so **executing the notebook
top to bottom IS the verification** — if any reference drifts, the cell
raises.

In [ ]:
from polaris_re.analytics.experience_validation import (
    run_experience_improvement_benchmarks,
)
from polaris_re.analytics.validation import (
    ValidationCategory,
    ValidationStatus,
    run_closed_form_benchmarks,
    run_full_validation_pack,
    run_statutory_deck_benchmarks,
)

report = run_full_validation_pack()
print(f"{report.n_passed}/{report.n_cases} cases passed "
      f"(all_passed={report.all_passed}).")

## Pass/fail table

The report renders itself as Markdown (identical to `polaris benchmark -o`).

In [ ]:
from IPython.display import Markdown, display

display(Markdown(report.to_markdown()))

## Diligence assertions

These are the checks that make the notebook self-verifying: every case
passes, all four reference categories are represented, and the sub-packs
compose into the full pack.

In [ ]:
# 1. Every reference is reproduced within its documented tolerance.
assert report.all_passed, (
    f"{report.n_failed} validation case(s) outside tolerance"
)
assert all(r.status is ValidationStatus.PASS for r in report.results)

# 2. The report spans all four reference categories (not just the cheap ones).
categories = {r.category for r in report.results}
assert categories == {
    ValidationCategory.CLOSED_FORM,
    ValidationCategory.TEXTBOOK,
    ValidationCategory.STATUTORY_DECK,
    ValidationCategory.EXPERIENCE_IMPROVEMENT,
}, categories

# 3. The full pack is exactly the union of the sub-packs (single source of truth).
closed_form = run_closed_form_benchmarks()
deck = run_statutory_deck_benchmarks()
experience = run_experience_improvement_benchmarks()
assert report.n_cases == closed_form.n_cases + deck.n_cases + experience.n_cases
assert closed_form.all_passed and deck.all_passed and experience.all_passed

print("All diligence assertions passed:")
print(f"  closed-form : {closed_form.n_passed}/{closed_form.n_cases}")
print(f"  deck        : {deck.n_passed}/{deck.n_cases}")
print(f"  experience  : {experience.n_passed}/{experience.n_cases}")
print(f"  full        : {report.n_passed}/{report.n_cases}")

In [ ]:
# Per-case detail: the tightest and loosest relative errors observed.
worst = max(report.results, key=lambda r: r.rel_error)
best = min(report.results, key=lambda r: r.rel_error)
print(f"Loosest match: {worst.name}")
print(f"  rel. error {worst.rel_error:.2e} within rtol {worst.tolerance_rtol:.1e}")
print(f"Tightest match: {best.name}")
print(f"  rel. error {best.rel_error:.2e}")

## Reproduce from the command line

The same pack is available headless and **exits non-zero on any failure**,
so it can gate a CI job:

```bash
polaris benchmark                    # full pack, pretty table, exit 1 on any FAIL
polaris benchmark --pack deck        # just the SOA Illustrative Life Table cases
polaris benchmark --pack experience  # just the MI-recovery identity cases
polaris benchmark -o report.md       # export the Markdown report above
polaris benchmark --json out.json    # export the structured results
```